# 🚦 Q-TrafficAI: Research Demonstration
### Adaptive Quantum Signal Allocation for AI-Driven Intelligent Traffic Optimization
---
This research notebook demonstrates the complete end-to-end AQSA workflow:
1. **Traffic Network Simulation**
2. **AI Traffic Prediction with Uncertainty Quantification**
3. **AQSA Signal Prioritization & Variable Selection**
4. **Dynamic QUBO Matrix Assembly**
5. **Ising Hamiltonian Mapping & QAOA Quantum Execution**
6. **Feasibility-Aware Solution Decoding**
7. **Comparative Baseline Benchmarking**

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.traffic.simulator import TrafficSimulator
from src.ai.predictor import TrafficPredictor
from src.aqsa.aqsa_pipeline import AQSAPipeline
from src.evaluation.comparison import AlgorithmComparator
import numpy as np
import pandas as pd

## 1. Initialize Simulator & Warm Up Traffic Grid
We initialize a 4-intersection network (Nodes A, B, C, D) under morning peak conditions.

In [ ]:
sim = TrafficSimulator(num_intersections=4, scenario='morning_peak', seed=42)
for _ in range(6):
    sim.step()

init_state = sim.get_state()
print(f"Initial Total Queued Vehicles: {init_state['total_queued']}")
print(f"Global Average Wait Time: {init_state['global_avg_wait']:.1f} s")

## 2. Load AI Predictor & Run AQSA Pipeline
AQSA prioritizes intersections using AI volume forecasts, queue severity, and prediction uncertainty.

In [ ]:
predictor = TrafficPredictor()
predictor.load('../models/traffic_predictor.joblib')

pipeline = AQSAPipeline(
    predictor=predictor,
    threshold=0.55,
    qubit_budget=6,
    qaoa_depth=1,
    shots=512
)

result = pipeline.run(init_state)
print(f"AQSA Run Completed in {result['execution_time_ms']:.1f} ms")
print('Selected Nodes for Quantum Allocation:', result['selection']['selected_node_ids'])
print('Active Qubits:', result['selection']['num_qubits'])
print('Best Feasible Bitstring:', result['decoding']['selected_solution']['bitstring'])

## 3. Dynamic QUBO Matrix Inspection
Inspect the dynamic upper-triangular QUBO matrix.

In [ ]:
qubo_mat = result['qubo']['matrix']
var_names = result['qubo']['variables']
df_qubo = pd.DataFrame(qubo_mat, index=var_names, columns=var_names)
print(df_qubo.round(2))

## 4. Multi-Algorithm Benchmarking
Evaluate Fixed, Greedy, Exact Brute Force, Standard QAOA, and AQSA+QAOA.

In [ ]:
comparator = AlgorithmComparator(predictor=predictor)
bench = comparator.run_comparison(num_intersections=4, scenario='morning_peak', sim_steps=10)

summary = []
for name, data in bench['results'].items():
    m = data['metrics']
    summary.append({
        'Algorithm': name,
        'Avg Wait (s)': m['avg_waiting_time_sec'],
        'Avg Queue': m['avg_queue_length_veh'],
        'Throughput': m['throughput_veh_per_min'],
        'Violations': m['constraint_violations'],
        'Runtime (ms)': m['computational_time_ms']
    })

df_bench = pd.DataFrame(summary)
print(df_bench.to_string(index=False))